# Lesson 04: Statistics & Probability on Real-World Data

In this notebook, you will:
1. Analyze Major League Baseball (MLB) player heights, weights, and roles.
2. Plot box plots by role and histograms of weight distributions.
3. Compute Student's $t$ confidence intervals ($p = 0.85, 0.90, 0.95$) and run `scipy.stats.ttest_ind` hypothesis tests.
4. Compute covariance and Pearson correlation matrices.

In [ ]:
!pip install -q numpy pandas scipy matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
# Self-contained synthetic MLB dataset matching SOCR role parameters
rng = np.random.default_rng(42)
role_specs = [
    ("Catcher", 72.72, 204.33, 76),
    ("Designated_Hitter", 74.22, 220.89, 18),
    ("First_Baseman", 74.00, 213.11, 55),
    ("Outfielder", 73.01, 199.11, 194),
    ("Relief_Pitcher", 74.37, 203.52, 315),
    ("Second_Baseman", 71.36, 184.34, 58),
    ("Shortstop", 71.90, 182.92, 52),
    ("Starting_Pitcher", 74.72, 205.16, 221),
    ("Third_Baseman", 73.04, 200.96, 45),
]
rows = []
for role, h_mean, w_mean, count in role_specs:
    h = rng.normal(h_mean, 2.1, count)
    w = w_mean + 4.5 * (h - h_mean) + rng.normal(0, 15.0, count)
    for hi, wi in zip(h, w):
        rows.append({"Role": role, "Height": round(hi, 2), "Weight": round(wi, 1)})

df = pd.DataFrame(rows)
print(df.groupby("Role")[["Height", "Weight"]].agg(["mean", "count"]).round(2))

In [ ]:
# Box plot by Role & Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.boxplot(column="Height", by="Role", ax=axes[0], rot=45)
axes[0].set_title("Player Height by Role")
axes[1].hist(df["Weight"], bins=25, color="#A7452A", edgecolor="white")
axes[1].set_title("Weight Distribution")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Confidence intervals & Welch's t-test
for p in [0.85, 0.90, 0.95]:
    ci = stats.t.interval(p, len(df)-1, loc=df["Weight"].mean(), scale=stats.sem(df["Weight"]))
    print(f"Weight {int(p*100)}% CI: {df['Weight'].mean():.2f} ({ci[0]:.2f} .. {ci[1]:.2f})")

fb = df.loc[df["Role"] == "First_Baseman", "Height"]
sb = df.loc[df["Role"] == "Second_Baseman", "Height"]
tval, pval = stats.ttest_ind(fb, sb, equal_var=False)
print(f"\nFirst vs Second Basemen Height t-test: t={tval:.2f}, p={pval:.2e}")
print("\nCorrelation Matrix (Height vs Weight):")
print(df[["Height", "Weight"]].corr().round(4))